In [1]:
# transformer-lens 3.4.0 pins torchvision>=0.22,<0.23 (for torch 2.7.x),
# but Colab has torch 2.10 + torchvision 0.25. This breaks pip's resolver.
# Workaround: install both with --no-deps, then install dependencies separately.

# Step 1: Install sae-lens and transformer-lens without the torchvision constraint
!pip install --no-deps transformer-lens sae-lens

# Step 2: Install all actual dependencies (minus torchvision, not needed for text SAEs)
!pip install \
    "transformers>=5.4.0,<6.0.0" \
    "datasets>=3.1.0" \
    "einops>=0.6.0" \
    "jaxtyping>=0.2.11" \
    "fancy-einsum>=0.0.3" \
    "beartype>=0.14.1" \
    "accelerate>=0.23.0" \
    "rich>=12.6.0" \
    "sentencepiece" \
    "typeguard>=4.2,<5" \
    "wandb>=0.13.5" \
    "transformers-stream-generator>=0.0.5,<0.1" \
    "better-abc>=0.0.3" \
    "protobuf>=3.20.0" \
    "huggingface-hub>=0.23.2" \
    "safetensors>=0.4.2,<1.0.0" \
    "plotly>=5.19.0" \
    "nltk>=3.8.1" \
    "python-dotenv>=1.0.1" \
    "pyyaml>=6.0.1" \
    "simple-parsing>=0.1.8" \
    "tenacity>=9.0.0" \
    "babe>=0.0.7"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 35.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 13.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.3/276.3 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.9/236.9 kB 29.1 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=79b148c2b053d1bab6eda22f36cd267289374339de418ea0ccea6c78be941575
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3

In [3]:
from huggingface_hub import login
import os
from __future__ import annotations

import pandas as pd
import argparse
import re
import sys
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Iterable

import requests
import torch, gc
from sae_lens import SAE
from transformer_lens import HookedTransformer

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive"
REPO_DIR = "/content/SAE_Surrogate"

gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))
print(torch.cuda.mem_get_info())
print(torch.cuda.is_bf16_supported())

Mounted at /content/drive
True
0
NVIDIA A100-SXM4-40GB
(41957130240, 42405855232)
True


In [4]:
with open(f"{DRIVE_DIR}/tokens/hftoken.txt") as f:
    token = f.readline().strip()

login(token=token)

In [5]:
with open(f"{DRIVE_DIR}/tokens/ghtoken.txt") as f:
    email = f.readline().strip()
    username = f.readline().strip()
    token = f.readline().strip()

!git config --global user.email "{email}"
!git config --global user.name "{username}"

#!git clone https://github.com/ntjohn04/SAE_Surrogate
if not os.path.exists(REPO_DIR):
    !git clone https://{token}@github.com/{username}/SAE_Surrogate
else:
    !git -C {REPO_DIR} pull

sys.path.append(f"{REPO_DIR}")

import util_neuronpedia
import util_nanogpt
import util_G8SMK

Cloning into 'SAE_Surrogate'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 32 (delta 11), reused 29 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 409.58 KiB | 19.50 MiB/s, done.
Resolving deltas: 100% (11/11), done.


In [6]:
DEFAULT_RELEASE = "gemma-scope-2b-pt-res"
DEFAULT_SAE_ID = "layer_20/width_16k/average_l0_71"
DEFAULT_MODEL = "gemma-2-2b"
NEURONPEDIA_FEATURE_API = "https://www.neuronpedia.org/api/feature"

DEVICE = 'cuda'
DTYPE = torch.bfloat16


In [8]:
print("----loading SAE----")
sae = SAE.from_pretrained(
    release=DEFAULT_RELEASE,
    sae_id=DEFAULT_SAE_ID,
    device=DEVICE,
    dtype=DTYPE
)

print("----loading feature catalog----")
if not os.path.exists(f"{REPO_DIR}/feature_catalog.csv"):
    print("\tbuilding feature catalog...")
    feature_catalog = util_neuronpedia.build_feature_catalog()
    feature_catalog.to_csv(f"{REPO_DIR}/feature_catalog.csv", index=False)
else:
    feature_catalog = pd.read_csv(f"{REPO_DIR}/feature_catalog.csv")

print("----loading hooks----")
metadata = getattr(sae.cfg, "metadata", None)
hook_name = getattr(metadata, "hook_name", None)
match = re.search(r"blocks\.(\d+)\.", hook_name)
hook_layer = int(match.group(1)) if match else None

model_name = getattr(metadata, "model_name", None)
prepend_bos = getattr(metadata, "prepend_bos", None)

print(f"\tHook name:\t{hook_name}")
print(f"\tHook layer:\t{hook_layer}")
print(f"\tModel name:\t{model_name}")
print(f"\tPrepend BOS:\t{prepend_bos}")

print("----loading model----")
model = HookedTransformer.from_pretrained_no_processing(model_name,
                                          device=DEVICE, 
                                          dtype=DTYPE, 
                                          default_prepend_bos=prepend_bos, 
                                          first_n_layers=hook_layer+1)

----loading SAE----
----loading feature catalog----
----loading hooks----
	Hook name:	blocks.20.hook_resid_post
	Hook layer:	20
	Model name:	gemma-2-2b
	Prepend BOS:	True
----loading model----


config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded pretrained model gemma-2-2b into HookedTransformer


In [9]:
def get_feature_sequence(sae, model, prepend_bos, hook_name, top_k, prompt):
    with torch.inference_mode():
        tokens = model.to_tokens(prompt, prepend_bos=prepend_bos)
        str_tokens = model.to_str_tokens(tokens[0])

        _, cache = model.run_with_cache(tokens, names_filter=[hook_name])
        activations = cache[hook_name]
        feature_acts = sae.encode(activations)

    return feature_acts, str_tokens

In [10]:
feature_acts, str_tokens = get_feature_sequence(sae, model, prepend_bos, hook_name, 5, "USA France London Australia Russia")

In [12]:
train_loader, test_loader, train_ds, test_ds = util_G8SMK.load_gsm8k_dataloaders(
    sae, model, prepend_bos, hook_name,
    cache_dir=f"{REPO_DIR}/sae_cache",
    max_length=256, batch_size=16,
)

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Precomputing SAE features: 100%|██████████| 935/935 [06:03<00:00,  2.58it/s]


Saved 7473 samples to /content/SAE_Surrogate/sae_cache/gsm8k_train_features.pt


Precomputing SAE features: 100%|██████████| 165/165 [01:05<00:00,  2.52it/s]


Saved 1319 samples to /content/SAE_Surrogate/sae_cache/gsm8k_test_features.pt


In [34]:
for i, (token_ids, feature_acts, mask) in enumerate(train_loader):
    print(f"Batch {i}:")
    print(f"\tToken IDs: {token_ids.shape}")
    print(f"\tFeature Activations: {feature_acts.shape}")
    print(f"\tMask: {mask.shape}")
    
    print(mask[0])
    break

Batch 0:
	Token IDs: torch.Size([16, 115])
	Feature Activations: torch.Size([16, 115, 16384])
	Mask: torch.Size([16, 115])
tensor([False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, Fals

In [29]:
util_G8SMK.print_dataset_summary(train_ds, "Train")


  Train
  Samples:          7473
  Feature dim:      16384
  Seq length:       min=0, max=213, mean=31.6, median=21
  At max length:    1 (0.0%)

  Feature activations (sampled from first 500 examples):
    Sparsity:       98.0% zeros
    Non-zero mean:  18.3573
    Non-zero max:   2040.0000
    Active features/token: mean=325.6, max=7017

  Question length (words): min=9, max=183, mean=45.1
  Parsed final answers: 7473/7473



In [35]:
util_G8SMK.print_samples(train_ds, indices=[0])


────────────────────────────────────────────────────────────
  Sample 0  (seq_len=0)
────────────────────────────────────────────────────────────

  Question:
    Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

  Answer:
    Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

  Final answer: 72

  Top 10 features (max activation across tokens):
     1. feature=8684    activation= 187.0000
     2. feature=5052    activation=  37.5000
     3. feature=1692    activation=  32.2500
     4. feature=7045    activation=  29.2500
     5. feature=8849    activation=  29.2500
     6. feature=9345    activation=  29.0000
     7. feature=10991   activation=  27.7500
     8. feature=15778   activation=  27.1250
     9. feature=13893   activation=  26.2500
    10. feature=743     activation=  26.0000
─────────────────

In [ ]:
def get_feature_acts(sae, model, hook_name: str, token_ids: torch.Tensor) -> torch.Tensor:
    """Forward a padded token batch through model + SAE.

    Parameters
    ----------
    token_ids : LongTensor [batch, seq_len]  (on model device)

    Returns
    -------
    feature_acts : Tensor [batch, seq_len, num_features]
    """
    with torch.inference_mode():
        _, cache = model.run_with_cache(token_ids, names_filter=[hook_name])
        feature_acts = sae.encode(cache[hook_name])
    return feature_acts


In [28]:
!git -C {REPO_DIR} pull

import importlib
importlib.reload(util_G8SMK)

import util_neuronpedia
import util_nanogpt
import util_G8SMK

Already up to date.


In [15]:
!git -C {REPO_DIR} add .
!git -C {REPO_DIR} commit -m "sae_cache"
!git -C {REPO_DIR} push origin main

^C
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	__pycache__/
	sae_cache/

nothing added to commit but untracked files present (use "git add" to track)
Everything up-to-date


In [18]:
for file in os.listdir(f"{REPO_DIR}/sae_cache"):
    print(file)
    print(f"file size: {os.path.getsize(os.path.join(REPO_DIR, 'sae_cache', file)) / (1024**3)} gigabytes")

gsm8k_test_features.pt
file size: 4.952468877658248 gigabytes
gsm8k_train_features.pt
file size: 27.309960260987282 gigabytes
